In [5]:
import pandas as pd
import json
import os
import re

def extract_storage_strategy(class_decl):
    """ฟังก์ชันช่วยดึงค่า StorageStrategy จากคำประกาศคลาส (Class Declaration)"""
    match = re.search(r'StorageStrategy\s*=\s*([a-zA-Z0-9_]+)', str(class_decl))
    return match.group(1) if match else "Default"

def prepare_full_data_dictionary():
    print("⏳ กำลังโหลดและประมวลผลข้อมูล CSV ทั้ง 5 ไฟล์...")
    
    # 1. โหลดข้อมูลทั้งหมด
    try:
        df_tables = pd.read_csv('iris_data_dict-sql_tables.csv').fillna('')
        df_fields = pd.read_csv('iris_data_dict-sql_fields.csv').fillna('')
        df_fk = pd.read_csv('iris_data_dict-fk_relationships.csv').fillna('')
        df_classes = pd.read_csv('iris_data_dict-classes.csv').fillna('')
        df_members = pd.read_csv('iris_data_dict-members.csv').fillna('')
    except FileNotFoundError as e:
        print(f"❌ ไม่พบไฟล์ CSV: {e}")
        return

    data_dictionary = []

    # 2. วนลูปประมวลผลตามรายชื่อตารางใน SQL
    for index, table_row in df_tables.iterrows():
        sql_table_name = str(table_row['sql_table_name']).strip()
        class_name = str(table_row['class_name']).strip()
        
        if not sql_table_name: continue
            
        # 3. ดึงข้อมูลระดับ Class จาก classes.csv
        class_info = df_classes[df_classes['class_name'] == class_name]
        storage_strategy = "Unknown"
        class_decl = ""
        
        if not class_info.empty:
            class_decl = str(class_info.iloc[0]['class_decl'])
            storage_strategy = extract_storage_strategy(class_decl)

        # โครงสร้างหลัก
        table_meta = {
            "tableName": sql_table_name,
            "className": class_name,
            "module": {
                "name": table_row['module_name'],
                "prefix": table_row['module_prefix']
            },
            "tableDescription": {
                "en": table_row['class_description'],
                "th": "" 
            },
            # 🌟 [เพิ่มใหม่] ข้อมูลเชิงลึกสำหรับ Developer / Ops
            "irisClassDetails": {
                "storageStrategy": storage_strategy,
                "parameters": [],
                "triggers": []
            },
            "columns": []
        }
        
        # 4. ดึงข้อมูล Members ที่ไม่ใช่ Property (เช่น Parameters, Triggers) จาก members.csv
        class_members = df_members[df_members['class_name'] == class_name]
        for _, member_row in class_members.iterrows():
            kind = str(member_row['member_kind']).lower()
            member_detail = {
                "name": member_row['member_name'],
                "value/type": member_row['member_type'] if member_row['member_type'] else member_row['member_decl'],
                "description": member_row['description']
            }
            if kind == 'parameter':
                table_meta["irisClassDetails"]["parameters"].append(member_detail)
            elif kind == 'trigger':
                table_meta["irisClassDetails"]["triggers"].append(member_detail)

        # 5. ดึงข้อมูล Columns (รวมถึง Object Reference)
        table_fields = df_fields[df_fields['class_name'] == class_name]
        for _, field_row in table_fields.iterrows():
            sql_field_name = str(field_row['sql_field_name']).strip()
            if not sql_field_name: continue
                
            fk_match = df_fk[(df_fk['source_class_name'] == class_name) & 
                             (df_fk['source_sql_field_name'] == sql_field_name)]
            is_reference = not fk_match.empty
            
            column_meta = {
                "name": sql_field_name,
                "dataType": field_row['member_type'],
                "description": {
                    "en": field_row['description'],
                    "th": "" 
                },
                "isReference": is_reference
            }
            if is_reference:
                target_table = str(fk_match.iloc[0]['target_sql_table_name'])
                column_meta["referenceTarget"] = target_table
                column_meta["arrowSyntaxHelp"] = f"{sql_field_name}->[Field]"
                
            table_meta["columns"].append(column_meta)
            
        data_dictionary.append(table_meta)

    # 6. บันทึกไฟล์ JSON
    output_filename = 'iris_data_dictionary_full.json'
    with open(output_filename, 'w', encoding='utf-8') as f:
        json.dump(data_dictionary, f, ensure_ascii=False, indent=2)

    print(f"✅ ประมวลผลสำเร็จ! นำข้อมูลจาก 5 ไฟล์มารวมกันเรียบร้อย")
    print(f"📁 บันทึกไฟล์ที่: {os.path.abspath(output_filename)}")

if __name__ == "__main__":
    prepare_full_data_dictionary()

⏳ กำลังโหลดและประมวลผลข้อมูล CSV ทั้ง 5 ไฟล์...
✅ ประมวลผลสำเร็จ! นำข้อมูลจาก 5 ไฟล์มารวมกันเรียบร้อย
📁 บันทึกไฟล์ที่: f:\Script\IRIS_Meta_Data\iris_data_dictionary_full.json
